In [52]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

In [53]:
load_dotenv(override=True)
OPEN_API_KEY = os.getenv("OPENAI_API_KEY")
if OPEN_API_KEY and OPEN_API_KEY.startswith("sk-proj-") and len(OPEN_API_KEY)>10:
    print("API key exists in env")
else:
    print("There might be a problem with your API key")
model = "gpt-5-nano"
openai = OpenAI()

API key exists in env


In [54]:
# Profile text is read only from `linkedin_profiles/<vanity>.txt` (vanity = LinkedIn /in/ slug).

from pathlib import Path
from urllib.parse import urlparse
from typing import Optional

model_system_prompt = """
You are a resume writer. Given LinkedIn profile content, produce an ATS-friendly resume.

Output only valid Markdown (no HTML):
- Use ## for main sections (e.g. Professional Summary, Technical Skills, Experience, Education, Certifications).
- Use **bold** for job titles, company names, and degree names.
- Use bullet lists (-) for skills, responsibilities, and achievements.
- Start with a single # line for the candidate name and headline or target role.
"""


def linkedin_vanity_from_url(url: str) -> Optional[str]:
    """Extract the /in/<vanity> slug from a LinkedIn profile URL, or None."""
    try:
        p = urlparse(url.strip())
        host = (p.netloc or "").lower()
        if "linkedin.com" not in host:
            return None
        parts = [x for x in p.path.split("/") if x]
        if len(parts) >= 2 and parts[0].lower() == "in":
            return parts[1].rstrip("/")
    except Exception:
        pass
    return None


LINKEDIN_PROFILES_DIR = Path("linkedin_profiles")


def load_linkedin_profile_content(url: str) -> str:
    """Read profile text from `linkedin_profiles/<vanity>.txt` (vanity parsed from the LinkedIn /in/ URL)."""
    vanity = linkedin_vanity_from_url(url)
    if not vanity:
        raise ValueError(
            f"Not a LinkedIn profile URL (expected ...linkedin.com/in/<vanity>): {url!r}"
        )
    path = LINKEDIN_PROFILES_DIR / f"{vanity}.txt"
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing profile file: {path.resolve()}\n"
            f"Create it and paste exported/copied profile text for '{vanity}'."
        )
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError(f"Profile file is empty: {path}")
    return text


def get_user_prompt(url: str) -> str:
    profile_text = load_linkedin_profile_content(url)
    return f"""LinkedIn profile URL: {url}

Use the following profile content to prepare an ATS-ready data engineer resume:

{profile_text}
"""


def display_profile_preview(url: str) -> str:
    """Show loaded profile text in the notebook as readable Markdown, and return the raw string."""
    content = load_linkedin_profile_content(url)
    fence = "````"
    display(
        Markdown(
            f"### Profile input loaded\n\n**URL:** `{url}`\n\n{fence}text\n{content}\n{fence}"
        )
    )
    return content

In [55]:
# Preview what will be sent to the model (Markdown: heading + fenced text block).
display_profile_preview("https://www.linkedin.com/in/vittal-manikonda")

### Profile input loaded

**URL:** `https://www.linkedin.com/in/vittal-manikonda`

````text
First Name,Last Name,Maiden Name,Address,Birth Date,Headline,Summary,Industry,Zip Code,Geo Location,Twitter Handles,Websites,Instant Messengers
Vittal,Manikonda,,,,Lead Software Engineer at Epsilon| Python | Shell Script | PySpark | GenAI | Microsoft/Databricks certified DataEngineer|Certified Databricks GENAI Fundamentals| Certified Databricks GENAI Associate ,"seasoned software professional with over 18 years of experience, Proficient in designing and implementing robust frameworks for data engineering projects, and leveraging extensive data engineering experience to solve complex business challenges, holding a Post Graduate Diploma in Data Science, with a strong passion for data analytics and a proven track record in end-to-end software development life cycle.  TECHNICAL SKILLS  Languages & Frameworks: Expertise in Python,SQL,REST Automation KEY SKILLS  •	Python	•	AWS	                • 	AZURE •	SQL          •	Databricks 	•	Shell scripting  EDUCATION  •	Post Graduate Diploma in Data Science | IIIT Bangalore & upGrad | Jun ’18 – May ‘19 •	M.C.A – Masters in Computer Applications | Sri Krishnadevaraya University, Bangalore | Karnataka, IN | Aug ’02 – Aug ‘05",Software Development,,"Bengaluru, Karnataka, India",,,
````

'First Name,Last Name,Maiden Name,Address,Birth Date,Headline,Summary,Industry,Zip Code,Geo Location,Twitter Handles,Websites,Instant Messengers\nVittal,Manikonda,,,,Lead Software Engineer at Epsilon| Python | Shell Script | PySpark | GenAI | Microsoft/Databricks certified DataEngineer|Certified Databricks GENAI Fundamentals| Certified Databricks GENAI Associate ,"seasoned software professional with over 18 years of experience, Proficient in designing and implementing robust frameworks for data engineering projects, and leveraging extensive data engineering experience to solve complex business challenges, holding a Post Graduate Diploma in Data Science, with a strong passion for data analytics and a proven track record in end-to-end software development life cycle.  TECHNICAL SKILLS  Languages & Frameworks: Expertise in Python,SQL,REST Automation KEY SKILLS  •\tPython\t•\tAWS\t                • \tAZURE •\tSQL          •\tDatabricks \t•\tShell scripting  EDUCATION  •\tPost Graduate Dipl

In [56]:
def create_resume(url: str, *, show_display: bool = True) -> str:
    print(f"Generating resume for {url} using model {model}")
    response = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": model_system_prompt},
            {"role": "user", "content": get_user_prompt(url)},
        ],
    )
    result = response.choices[0].message.content
    if not result:
        raise ValueError("Model returned empty content; check model name and API response.")
    if show_display:
        display(Markdown(result))
    return result


In [57]:
create_resume("https://www.linkedin.com/in/vittal-manikonda")

Generating resume for https://www.linkedin.com/in/vittal-manikonda using model gpt-5-nano


# Vittal Manikonda - Lead Data Engineer

## Professional Summary
Seasoned software professional with over 18 years of experience, proficient in designing and implementing robust frameworks for data engineering projects. Strong track record in end-to-end software development life cycle and solving complex business challenges through data analytics. Holds a **Post Graduate Diploma in Data Science** and certifications in Databricks GENAI Fundamentals/Associate and Microsoft/Databricks Data Engineer. Based in Bengaluru, Karnataka, India. Proficient in Python, SQL, REST Automation, PySpark, Shell scripting, with hands-on experience in AWS, Azure, and Databricks.

## Technical Skills
- Python
- SQL
- REST Automation
- PySpark
- Shell scripting
- AWS
- Azure
- Databricks
- GenAI

## Experience
- **Lead Software Engineer** — **Epsilon**
  - Design and implement robust data engineering frameworks to support data-driven decision making.
  - Lead end-to-end software development life cycle for data engineering projects.
  - Leverage data analytics to solve complex business challenges and drive data-informed decisions.
  - Collaborate with cross-functional teams to deliver scalable data pipelines and analytics solutions.

## Education
- **Post Graduate Diploma in Data Science** | IIIT Bangalore & upGrad | Jun ’18 – May ‘19
- **M.C.A – Masters in Computer Applications** | Sri Krishnadevaraya University, Bangalore | Aug ’02 – Aug ‘05

## Certifications
- Microsoft/Databricks certified DataEngineer
- Certified Databricks GENAI Fundamentals
- Certified Databricks GENAI Associate

'# Vittal Manikonda - Lead Data Engineer\n\n## Professional Summary\nSeasoned software professional with over 18 years of experience, proficient in designing and implementing robust frameworks for data engineering projects. Strong track record in end-to-end software development life cycle and solving complex business challenges through data analytics. Holds a **Post Graduate Diploma in Data Science** and certifications in Databricks GENAI Fundamentals/Associate and Microsoft/Databricks Data Engineer. Based in Bengaluru, Karnataka, India. Proficient in Python, SQL, REST Automation, PySpark, Shell scripting, with hands-on experience in AWS, Azure, and Databricks.\n\n## Technical Skills\n- Python\n- SQL\n- REST Automation\n- PySpark\n- Shell scripting\n- AWS\n- Azure\n- Databricks\n- GenAI\n\n## Experience\n- **Lead Software Engineer** — **Epsilon**\n  - Design and implement robust data engineering frameworks to support data-driven decision making.\n  - Lead end-to-end software developmen